# Malware classification with Qiskit Aer (IBM quantum simulator)

This notebook contains the core pipeline from
`Student_Work/hardware_results_experiment.py`: clean and select
features, create balanced train/test sets, fit preprocessing and principal component analysis (PCA) on the
training data, estimate a quantum fidelity kernel with Aer, and fit a support vector
machine (classical--SVM).

Start with CLaMP, or change `DATASET` to `mbr` or `custom` below. Download instructions
are in [Malware_Datasets/README.md](Malware_Datasets/README.md).
Install the packages with `python -m pip install -r requirements.txt` before opening
the notebook. Launch Jupyter from the repository root.

Defaults: **4 qubits, 512 shots, 80 training rows and 80 test rows**.
This runs one selected split on a local, noise-free simulator with finite shots.
The historical hardware table in [results/](results/README.md) averages three splits
and is a separate reference.


In [1]:
from collections import defaultdict
import hashlib
from importlib.metadata import version
from pathlib import Path
import platform
import re
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display
from qiskit import transpile
from qiskit.circuit import ParameterVector, QuantumCircuit
from qiskit.circuit.library import zz_feature_map
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSamplerV2
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

print('Python:', platform.python_version())
print({name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'qiskit', 'qiskit-aer']})


Python: 3.12.3
{'numpy': '2.5.1', 'pandas': '3.0.5', 'scikit-learn': '1.9.0', 'qiskit': '2.5.1', 'qiskit-aer': '0.17.2'}


## 1. Choose a dataset and circuit size

Each PCA component becomes one qubit. Change `N_QUBITS` to at least 2 and no more
than the number of input features or training rows. Larger samples require
quadratically more kernel evaluations; more qubits also increase simulation cost.

For your own CSV, set `DATASET = "custom"`, its path, label column and a list of
numeric feature columns below. Labels must be **0 = benign, 1 = malware**.
Exclude identifiers, hashes and any columns that reveal the label.
Missing feature values are allowed and will be imputed.


In [2]:
DATASET = 'clamp'           # 'clamp', 'mbr', or 'custom'; rerun all cells after changing
N_QUBITS = 4
SHOTS = 512
ROWS_PER_CLASS = 40        # Per class in EACH of training and test sets
FOLD = 0                   # Select 0, 1, or 2 from the original three-split protocol

CUSTOM_PATH = Path('Malware_Datasets/my_dataset.csv')
CUSTOM_LABEL = 'label'
CUSTOM_FEATURES = []        # Example: ['SizeOfCode', 'ImageBase', ...]

# Original experiment settings; keep fixed for comparable reruns.
MASTER_SEED = 20_260_825
TRAIN_SEEDS = (101, 204, 303)
N_FOLDS = len(TRAIN_SEEDS)
TRAIN_ROWS = TEST_ROWS = 2 * ROWS_PER_CLASS
REPS = 2
ENTANGLEMENT = 'linear'
SVC_C = 1.0
TRANSPILER_SEED = 42
AER_PARAMETER_CHUNK_SIZE = 128

ROOT = Path.cwd()
if not (ROOT / 'Malware_Datasets').is_dir():
    raise FileNotFoundError('Launch Jupyter from the repository root (the folder containing this notebook).')
if DATASET not in {'clamp', 'mbr', 'custom'}:
    raise ValueError("DATASET must be 'clamp', 'mbr', or 'custom'.")
if not 0 <= FOLD < N_FOLDS:
    raise ValueError('FOLD must be 0, 1, or 2.')
if not isinstance(N_QUBITS, int) or N_QUBITS < 2:
    raise ValueError('N_QUBITS must be an integer of at least 2 for the ZZ feature map.')
if not isinstance(ROWS_PER_CLASS, int) or ROWS_PER_CLASS < 1 or not isinstance(SHOTS, int) or SHOTS < 1:
    raise ValueError('ROWS_PER_CLASS and SHOTS must be positive integers.')

CLAMP_FEATURES = [
    'NumberOfSections', 'Characteristics', 'Magic', 'MajorLinkerVersion',
    'SizeOfCode', 'AddressOfEntryPoint', 'ImageBase', 'SizeOfImage',
    'DllCharacteristics', 'SizeOfStackReserve',
]
MBR_FEATURES = [
    'SizeOfCode', 'SizeOfInitializedData', 'SizeOfUninitializedData',
    'AddressOfEntryPoint', 'BaseOfData', 'ImageBase', 'NumberOfSections', 'DllCharacteristics',
]
presets = {
    'clamp': (ROOT / 'Malware_Datasets/ClaMP_Raw-5184.csv', 'class', CLAMP_FEATURES),
    'mbr': (ROOT / 'Malware_Datasets/data/fabiana/Malware_and_benign_recognition.csv', 'Malicious', MBR_FEATURES),
    'custom': (CUSTOM_PATH, CUSTOM_LABEL, CUSTOM_FEATURES),
}
data_path, label_column, feature_columns = presets[DATASET]
# Preserve the original MBR seed names and row identifiers.
dataset_id = 'fabiana' if DATASET == 'mbr' else DATASET


## 2. Load, clean and remove duplicate feature vectors

Keep the existing feature selections for CLaMP and MBR. Convert infinite values
to missing values. Keep one row for each repeated feature vector; discard vectors
with conflicting labels. This happens before splitting so duplicate rows cannot
appear in both training and test data. The helper functions below are retained
from the original experiment.


In [3]:
def canonical_row_bytes(row: np.ndarray) -> bytes:
    values = np.asarray(row, dtype="<f8").copy()
    values[np.isnan(values)] = np.nan
    values[values == 0.0] = 0.0
    return values.tobytes(order="C")

def row_hashes(values: np.ndarray) -> np.ndarray:
    return np.asarray(
        [hashlib.sha256(canonical_row_bytes(row)).hexdigest() for row in values],
        dtype="U64",
    )

def _deduplicate_pool(
    x: np.ndarray,
    y: np.ndarray,
    source_ids: np.ndarray,
    architectures: np.ndarray | None,
) -> tuple[dict[str, np.ndarray], dict[str, Any]]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=np.int8)
    source_ids = np.asarray(source_ids, dtype=str)
    architectures = (
        np.full(len(y), "", dtype="U1")
        if architectures is None
        else np.asarray(architectures, dtype=str)
    )
    hashes = row_hashes(x)
    groups: dict[str, list[int]] = defaultdict(list)
    for index, digest in enumerate(hashes):
        groups[str(digest)].append(index)

    keep: list[int] = []
    conflict_groups = 0
    conflict_rows = 0
    duplicate_rows_removed = 0
    for digest in sorted(groups):
        indices = groups[digest]
        labels = np.unique(y[indices])
        if len(labels) != 1:
            conflict_groups += 1
            conflict_rows += len(indices)
            continue
        representative = min(indices, key=lambda idx: source_ids[idx])
        keep.append(representative)
        duplicate_rows_removed += len(indices) - 1

    keep_array = np.asarray(keep, dtype=int)
    order = np.argsort(source_ids[keep_array], kind="stable")
    keep_array = keep_array[order]
    pool = {
        "x": x[keep_array],
        "y": y[keep_array],
        "source_ids": source_ids[keep_array],
        "row_hashes": hashes[keep_array],
        "architectures": architectures[keep_array],
    }
    summary = {
        "candidate_rows": len(y),
        "unique_clean_groups": len(keep_array),
        "duplicate_rows_removed": duplicate_rows_removed,
        "conflicting_groups_excluded": conflict_groups,
        "conflicting_rows_excluded": conflict_rows,
        "class_counts": {
            str(int(label)): int(count)
            for label, count in zip(*np.unique(pool["y"], return_counts=True))
        },
    }
    return pool, summary

if not data_path.is_file():
    raise FileNotFoundError(f'Missing {data_path}. See Malware_Datasets/README.md for downloads.')
if not feature_columns or len(set(feature_columns)) != len(feature_columns) or label_column in feature_columns:
    raise ValueError('Provide a nonempty list of distinct numeric feature columns, excluding the label.')
frame = pd.read_csv(data_path)
missing = [name for name in [*feature_columns, label_column] if name not in frame]
if missing:
    raise ValueError(f'Missing CSV columns: {missing}')
if frame[label_column].isna().any() or set(frame[label_column].unique()) != {0, 1}:
    raise ValueError('The label column must contain both 0 (benign) and 1 (malware), with no missing labels.')
x = frame[feature_columns].replace([np.inf, -np.inf], np.nan).to_numpy(dtype=float)
y = frame[label_column].to_numpy(dtype=np.int8)
source_ids = np.asarray([f'{dataset_id}:{index}' for index in frame.index], dtype=str)
pool, cleaning_summary = _deduplicate_pool(x, y, source_ids, None)
dataset_sha256 = hashlib.sha256(data_path.read_bytes()).hexdigest()
print(f'Loaded {len(frame):,} rows and {len(feature_columns)} features from {data_path.name}')
print(cleaning_summary)


Loaded 5,184 rows and 10 features from ClaMP_Raw-5184.csv
{'candidate_rows': 5184, 'unique_clean_groups': 4040, 'duplicate_rows_removed': 1144, 'conflicting_groups_excluded': 0, 'conflicting_rows_excluded': 0, 'class_counts': {'0': 2275, '1': 1765}}


## 3. Select a balanced train/test split

Reuse the original deterministic sampling: reserve three disjoint test sets and
sample training rows from the remainder. `FOLD` selects which split to run.
Each class needs at least `4 * ROWS_PER_CLASS` distinct, nonconflicting vectors
(160 by default); reduce `ROWS_PER_CLASS` for smaller custom datasets.


In [4]:
def stable_seed(name: str, base: int = MASTER_SEED) -> int:
    digest = hashlib.sha256(f"{base}:{name}".encode()).digest()
    return int.from_bytes(digest[:4], "little")

folds = [{'train': np.array([], dtype=int), 'test': np.array([], dtype=int)} for _ in range(N_FOLDS)]
for label in (0, 1):
    indices = np.flatnonzero(pool['y'] == label)
    needed = (N_FOLDS + 1) * ROWS_PER_CLASS
    if len(indices) < needed:
        raise ValueError(f'Class {label} has {len(indices)} unique rows; need {needed}. Reduce ROWS_PER_CLASS.')
    rng = np.random.default_rng(stable_seed(f'{dataset_id}:test:{label}'))
    indices = indices[rng.permutation(len(indices))]
    reserved_test = indices[:N_FOLDS * ROWS_PER_CLASS]
    eligible_train = indices[N_FOLDS * ROWS_PER_CLASS:]
    for fold in range(N_FOLDS):
        test = reserved_test[fold * ROWS_PER_CLASS:(fold + 1) * ROWS_PER_CLASS]
        train_rng = np.random.default_rng(stable_seed(f'{dataset_id}:train:{label}', TRAIN_SEEDS[fold]))
        train = train_rng.choice(eligible_train, size=ROWS_PER_CLASS, replace=False)
        folds[fold]['test'] = np.concatenate([folds[fold]['test'], test])
        folds[fold]['train'] = np.concatenate([folds[fold]['train'], train])

train_indices, test_indices = folds[FOLD]['train'], folds[FOLD]['test']
assert not set(train_indices) & set(np.concatenate([fold['test'] for fold in folds]))
raw_train, raw_test = pool['x'][train_indices], pool['x'][test_indices]
y_train, y_test = pool['y'][train_indices], pool['y'][test_indices]
print(f'Split {FOLD}: {len(y_train)} training rows, {len(y_test)} test rows; {ROWS_PER_CLASS} per class in each.')


Split 0: 80 training rows, 80 test rows; 40 per class in each.


## 4. Fit preprocessing and PCA on training data only

Replace missing values with the training median, standardize features, reduce
them to `N_QUBITS` PCA components, and scale each component to `[0, 1]` for the
feature map. Apply those fitted transforms to the test rows, clipping values
outside the training range. No test data is used to fit any transform.


In [5]:
if N_QUBITS > min(raw_train.shape):
    raise ValueError(f'N_QUBITS must be at most min(training rows, input features) = {min(raw_train.shape)}.')

imputer = SimpleImputer(strategy='median', keep_empty_features=True)
train_imputed = imputer.fit_transform(raw_train)
test_imputed = imputer.transform(raw_test)

standard = StandardScaler()
train_standard = standard.fit_transform(train_imputed)
test_standard = standard.transform(test_imputed)

pca = PCA(n_components=N_QUBITS, svd_solver='full')
train_representation = pca.fit_transform(train_standard)
test_representation = pca.transform(test_standard)

range_scaler = MinMaxScaler(feature_range=(0, 1), clip=True)
x_train = range_scaler.fit_transform(train_representation)
x_test = range_scaler.transform(test_representation)
if not np.isfinite(x_train).all() or not np.isfinite(x_test).all():
    raise ValueError('Preprocessing produced non-finite quantum inputs.')
# Retain the original guard against train/test collisions introduced by compression or clipping.
matches = np.all(x_test[:, np.newaxis, :] == x_train[np.newaxis, :, :], axis=2)
if matches.any():
    raise ValueError('Transformed train/test vectors overlap. Try more PCA components or a different FOLD.')
print(f'PCA variance retained: {pca.explained_variance_ratio_.sum():.1%}')
print(f'Quantum input shapes: train {x_train.shape}, test {x_test.shape}')


PCA variance retained: 67.0%
Quantum input shapes: train (80, 4), test (80, 4)


## 5. Estimate the quantum kernel with Qiskit Aer

The original two-repetition, linearly entangled ZZ feature map encodes each row.
For each pair, compose one encoded circuit with the inverse of the other. The
fraction of all-zero measurements estimates their squared overlap (fidelity).
Build a symmetric training kernel with diagonal 1, and a test-versus-training kernel.
These helpers retain the original batched Aer sampling and seeded shot counts.

At the default sample size there are 9,560 circuit evaluations, each with 512
shots. This step may take a few minutes; progress prints as batches finish.


In [6]:
def train_test_pairs(n_train: int, n_test: int) -> tuple[np.ndarray, np.ndarray]:
    train_pairs = np.asarray(
        [(i, j) for i in range(n_train) for j in range(i + 1, n_train)],
        dtype=np.int32,
    )
    test_pairs = np.asarray(
        [(i, j) for i in range(n_test) for j in range(n_train)],
        dtype=np.int32,
    )
    return train_pairs, test_pairs

def _fidelity_circuit(width: int) -> tuple[QuantumCircuit, ParameterVector, ParameterVector]:
    left_parameters = ParameterVector("left", width)
    right_parameters = ParameterVector("right", width)
    left = zz_feature_map(
        feature_dimension=width, reps=REPS, entanglement=ENTANGLEMENT
    ).assign_parameters(left_parameters)
    right = zz_feature_map(
        feature_dimension=width, reps=REPS, entanglement=ENTANGLEMENT
    ).assign_parameters(right_parameters)
    circuit = left.compose(right.inverse())
    circuit.measure_all()
    return circuit, left_parameters, right_parameters

_PARAMETER_PATTERN = re.compile(r'^(left|right)\[(\d+)]$')

def parameter_values(
    circuit: QuantumCircuit,
    left_values: np.ndarray,
    right_values: np.ndarray,
) -> np.ndarray:
    if len(left_values) != len(right_values):
        raise ValueError("Left/right parameter batches differ")
    columns: list[np.ndarray] = []
    for parameter in circuit.parameters:
        match = _PARAMETER_PATTERN.match(parameter.name)
        if not match:
            raise ValueError(f"Unexpected parameter name {parameter.name!r}")
        side, index_text = match.groups()
        values = left_values if side == "left" else right_values
        columns.append(values[:, int(index_text)])
    return np.column_stack(columns)

def _fidelities(pub_result: Any, expected: int) -> tuple[np.ndarray, int]:
    bit_array = pub_result.join_data()
    if bit_array.size != expected:
        raise RuntimeError(f"Expected {expected} results; received {bit_array.size}")
    shots = int(bit_array.num_shots)
    values = np.asarray(
        [bit_array.get_int_counts(index).get(0, 0) / shots for index in range(expected)],
        dtype=float,
    )
    return values, shots

def kernels_from_fidelities(
    train_fidelities: np.ndarray,
    test_fidelities: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    train_pairs, test_pairs = train_test_pairs(TRAIN_ROWS, TEST_ROWS)
    if len(train_fidelities) != len(train_pairs) or len(test_fidelities) != len(test_pairs):
        raise ValueError("Fidelity result length mismatch")
    train = np.eye(TRAIN_ROWS, dtype=float)
    for value, (left, right) in zip(train_fidelities, train_pairs):
        train[left, right] = value
        train[right, left] = value
    test = np.empty((TEST_ROWS, TRAIN_ROWS), dtype=float)
    for value, (left, right) in zip(test_fidelities, test_pairs):
        test[left, right] = value
    return train, test

def _run_aer_pubs(
    simulator: AerSimulator,
    circuit: QuantumCircuit,
    train_values: np.ndarray,
    test_values: np.ndarray,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    def run_chunks(values: np.ndarray, series: str) -> np.ndarray:
        chunks: list[np.ndarray] = []
        total = len(values)
        for start in range(0, total, AER_PARAMETER_CHUNK_SIZE):
            end = min(start + AER_PARAMETER_CHUNK_SIZE, total)
            chunk_seed = stable_seed(
                f"aer:{seed}:{series}:{start}:{end}", base=seed
            )
            sampler = AerSamplerV2.from_backend(simulator, seed=chunk_seed)
            result = sampler.run(
                [(circuit, values[start:end])], shots=SHOTS
            ).result()
            fidelities, observed_shots = _fidelities(result[0], end - start)
            if observed_shots != SHOTS:
                raise RuntimeError("Aer returned an unexpected shot count")
            chunks.append(fidelities)
            if end == total or end % (8 * AER_PARAMETER_CHUNK_SIZE) == 0:
                print(f"  Aer {series}: {end:,}/{total:,} fidelities", flush=True)
        return np.concatenate(chunks)

    train_fidelities = run_chunks(train_values, "train")
    test_fidelities = run_chunks(test_values, "test")
    return kernels_from_fidelities(train_fidelities, test_fidelities)

# Limit threads to keep the small example comfortable on a laptop.
simulator = AerSimulator(method='statevector', max_parallel_threads=2)
raw_circuit, _, _ = _fidelity_circuit(N_QUBITS)
circuit = transpile(raw_circuit, simulator, optimization_level=3, seed_transpiler=TRANSPILER_SEED)
train_pairs, test_pairs = train_test_pairs(TRAIN_ROWS, TEST_ROWS)
train_values = parameter_values(circuit, x_train[train_pairs[:, 0]], x_train[train_pairs[:, 1]])
test_values = parameter_values(circuit, x_test[test_pairs[:, 0]], x_train[test_pairs[:, 1]])
simulator_seed = stable_seed(f'{dataset_id}_pca{N_QUBITS}:fold{FOLD}:ideal_aer')
raw_train_kernel, test_kernel = _run_aer_pubs(simulator, circuit, train_values, test_values, simulator_seed)
print(f'Kernel shapes: train {raw_train_kernel.shape}, test {test_kernel.shape}')


  Aer train: 1,024/3,160 fidelities


  Aer train: 2,048/3,160 fidelities


  Aer train: 3,072/3,160 fidelities


  Aer train: 3,160/3,160 fidelities


  Aer test: 1,024/6,400 fidelities


...

  Aer test: 4,096/6,400 fidelities


  Aer test: 5,120/6,400 fidelities


  Aer test: 6,144/6,400 fidelities


  Aer test: 6,400/6,400 fidelities


Kernel shapes: train (80, 80), test (80, 80)


## 6. Train the SVM and save core metrics

Finite shots can make the training kernel slightly indefinite. Keep the existing
eigenvalue clipping and normalization before fitting `SVC(kernel="precomputed", C=1)`.
Report accuracy, balanced accuracy (mean recall across classes), and ROC AUC on
the held-out test rows. Confusion-matrix rows are true labels; columns are predictions.

Save one compact CSV in `results/`. Rerunning the same dataset/qubit/split setting
replaces its notebook CSV; the historical `reported_metrics.csv` stays separate.
These small-sample results do not establish full-dataset performance or quantum advantage.


In [7]:
def _project_psd(matrix: np.ndarray) -> tuple[np.ndarray, dict[str, float]]:
    symmetric = (np.asarray(matrix, dtype=float) + np.asarray(matrix, dtype=float).T) / 2
    np.fill_diagonal(symmetric, 1.0)
    values, vectors = np.linalg.eigh(symmetric)
    clipped = np.clip(values, 0, None)
    projected = (vectors * clipped) @ vectors.T
    diagonal = np.sqrt(np.clip(np.diag(projected), 1e-15, None))
    projected = projected / np.outer(diagonal, diagonal)
    projected = (projected + projected.T) / 2
    np.fill_diagonal(projected, 1.0)
    diagnostics = {
        "raw_minimum_eigenvalue": float(values.min()),
        "negative_eigenvalue_count": int(np.sum(values < -1e-12)),
        "negative_eigenvalue_mass": float(np.abs(values[values < 0]).sum()),
        "projected_minimum_eigenvalue": float(np.linalg.eigvalsh(projected).min()),
    }
    return projected, diagnostics

def _classification_metrics(
    k_train: np.ndarray,
    k_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
) -> tuple[dict[str, Any], np.ndarray, np.ndarray]:
    model = SVC(kernel="precomputed", C=SVC_C)
    model.fit(k_train, y_train)
    predictions = model.predict(k_test)
    scores = model.decision_function(k_test)
    metrics = {
        "accuracy": float(accuracy_score(y_test, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, predictions)),
        "roc_auc": float(roc_auc_score(y_test, scores)),
        "confusion_matrix": confusion_matrix(y_test, predictions, labels=[0, 1]),
        "benign_recall": float(np.mean(predictions[y_test == 0] == 0)),
        "malware_recall": float(np.mean(predictions[y_test == 1] == 1)),
    }
    return metrics, predictions.astype(np.int8), scores.astype(float)

train_kernel, psd_diagnostics = _project_psd(raw_train_kernel)
metrics, predictions, scores = _classification_metrics(train_kernel, test_kernel, y_train, y_test)
summary = pd.DataFrame([{
    'dataset': DATASET,
    'dataset_sha256': dataset_sha256,
    'qubits': N_QUBITS,
    'fold': FOLD,
    'master_seed': MASTER_SEED,
    'train_seed': TRAIN_SEEDS[FOLD],
    'simulator_seed': simulator_seed,
    'shots': SHOTS,
    'train_rows': TRAIN_ROWS,
    'test_rows': TEST_ROWS,
    'accuracy': metrics['accuracy'],
    'balanced_accuracy': metrics['balanced_accuracy'],
    'roc_auc': metrics['roc_auc'],
}])
display(summary[['dataset', 'qubits', 'fold', 'accuracy', 'balanced_accuracy', 'roc_auc']])
display(pd.DataFrame(metrics['confusion_matrix'], index=['true benign', 'true malware'], columns=['predicted benign', 'predicted malware']))
output_dir = ROOT / 'results'
output_dir.mkdir(exist_ok=True)
output_path = output_dir / f'notebook_{DATASET}_pca{N_QUBITS}_fold{FOLD}.csv'
summary.to_csv(output_path, index=False)
print(f'Saved {output_path.relative_to(ROOT)}')


,dataset,qubits,fold,accuracy,balanced_accuracy,roc_auc
0,clamp,4,0,0.775,0.775,0.82375


,predicted benign,predicted malware
true benign,35,5
true malware,13,27


Saved results/notebook_clamp_pca4_fold0.csv


To try another dataset, qubit count or split, change the settings and **Restart
Kernel and Run All**. Fixed seeds and the pinned environment make local reruns
repeatable. Different circuit compilation or package versions can change finite-shot
results, so this notebook does not promise to exactly reproduce archived Aer values.

For a separate hardware extension, see IBM's
[Run your first circuit on hardware](https://quantum.cloud.ibm.com/docs/en/guides/hello-world).
The original hardware campaign and full analyses are preserved in `Student_Work/`.
